In [2]:
import os, socket, sys
print(socket.gethostname(), os.environ.get("SLURM_JOB_ID"), os.environ.get("SLURM_CPUS_PER_TASK"))
print(sys.executable)
print(os.getcwd())


# Probe the training paradigms: fixed implementation

Checks `core/rl/` (the RL layer rebuilt on RLlib's own algorithms and modules)
with Andy Jones's probe environments
([Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html)),
made multi-agent and continuous in `core/rl/envs/probes.py`.

**The idea.** Each probe is a tiny environment whose correct answer is known
exactly, and each adds one piece of the learning problem to the previous one. Run
in order, the first probe a combination fails names the broken piece.

**How a probe is judged.** Not by the return (that mixes in exploration noise),
but by reading the trained networks in fixed states: the value (`V` for PPO and
APPO, the smaller twin `Q` for SAC) and the most likely action, compared with the
exact answer. A run passes when every action is within 0.15 and every value within
15 % (at least 0.15) of the answer. In CTCE the group's value is compared with the sum of the agents' values, since its reward is the sum; in DTDE and CTDE each agent's value with its own.

**How to use it.**

1. Run the setup cell, then choose what to test in **section b**: algorithms,
   paradigms, and the preset (`"fast"` for day-to-day checks, `"thorough"` for a
   slower check with the larger network and longer budgets).
2. Run the probe sections in order (c to g). Each trains only the selected
   combinations on that probe and shows the result right below; one combination
   on one probe takes seconds to a minute with `"fast"`.
3. Section h summarises everything selected; section i shows the learners' health.

Runs are written to `runs/probes_fast/` or `runs/probes_thorough/` (not in git)
and reused: running a cell again trains only what has not finished. Set
`RETRAIN = True` to train the selection again from scratch.

In [3]:
import json
import os
import shutil
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

# Repository root: found from wherever the notebook runs, and put on PYTHONPATH.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "rl").is_dir())
sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(ROOT), os.environ.get("PYTHONPATH", "")]))  # the training processes import core.rl too

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ray.rllib.core.columns import Columns

from core.rl.envs.probes import (
    ActionProbe,
    ConstantProbe,
    DiscountProbe,
    ObsActionProbe,
    ObservationProbe,
)
from core.rl import Run, load_policy, train
from core.rl.grouping import GROUP, central_obs, join

pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)

## a. The five probes

Three agents, each with a one-number observation and action in [-1, 1] and its own
reward. In probes 4 and 5 the agents are named `agent_2, agent_10, agent_1`, so
their sorted order differs from their order in the environment, and each agent has
a different target: an action delivered to the wrong agent shows as a wrong answer.

| Probe | Observation | Reward | Length | Exact answer | Adds | If it is the first to fail |
|---|---|---|---|---|---|---|
| 1 Constant | 0 | +1 | 1 step | V = 1 (group 3) | value learning, reward delivery, episode end | value loss, optimiser or reward delivery |
| 2 Observation | ±1, random per agent | the agent's observation | 1 step | V(o) = o (group: the sum) | the observation reaches the value network, per agent | observations do not reach the critic, or reach the wrong one |
| 3 Discount | 0, then 1 | +1 at the end only | 2 steps | V(1) = 1, V(0) = γ (γ = 0.5) | time: discounting, bootstrapping, episode boundary | the return calculation or the end-of-episode handling |
| 4 Action | 0 | 1 − (a − t)², targets −0.6, 0, 0.6 | 1 step | action = the agent's target | the policy update, each agent's action reaching its own reward | the actor's update path or the routing of actions |
| 5 Obs → action | ±1, random per agent | 1 − (a − o/2)² | 1 step | action = o/2 | the policy using its own observation | the actor ignores its input, or batches misalign observation, action and reward |

Probes 1-3 ignore the action, so they test the value side alone; 4 and 5 add the
policy. For a Gaussian policy the expected reward of probes 4 and 5 is
1 − (μ − t)² − σ², highest at μ = t for any spread σ, so the mean is a clean
read-out while the policy still explores.

**SAC's value in probe 3.** SAC learns a soft value that also counts the entropy
bonus it expects to collect, so its correct first-step value is γ·(1 + α·H) per
agent, with α the learned temperature and H the policy's entropy at the second
step. The checks below use that answer for SAC.

## b. Choose what to test

- `ALGORITHMS`, `PARADIGMS`: any of `"ppo", "appo", "sac"` and
  `"ctce", "ctde", "dtde"`; every combination of the two is trained.
- `PRESET`: `"fast"` (a 2 × 64 network, short budgets, more updates per step;
  seconds to a minute per run) or `"thorough"` (RLlib's default 2 × 256 network
  and budgets several times longer). The fast budgets are a starting point: if a
  run fails, check with a longer budget (section j) before reading it as a
  defect. Both start from `core/rl/config.SETTINGS`
  (the paper's hyperparameters) with a learning rate the probes can learn with in
  this budget; the budget is the same for every paradigm within an algorithm.
- `WORKERS`: runs trained in parallel processes (each loads torch and Ray).

In [4]:
ALGORITHMS = ["ppo"]  # V2: PPO only (APPO and SAC entries below are V1 settings)
PARADIGMS = ["ctce","ctde","dtde"]
PRESET = "thorough"
WORKERS = 45
RETRAIN = False  # True: train the selection again, even where a run has finished
SEED = 0
TOLERANCE = 0.15

PRESETS = {
    "fast": {
        "model": {"fcnet_hiddens": [64, 64]},
        # environment steps per run, by probe
        "budget": {
            "ppo": {1: 3_000, 2: 3_000, 3: 3_000, 4: 8_000, 5: 15_000},
            "appo": {1: 3_000, 2: 3_000, 3: 3_000, 4: 8_000, 5: 15_000},
            "sac": {1: 2_000, 2: 2_000, 3: 2_000, 4: 4_000, 5: 8_000},
        },
        "training": {
            "ppo": {"lr": 1e-3, "train_batch_size_per_learner": 500,
                    "minibatch_size": 64, "num_epochs": 10},
            "appo": {"lr": 1e-3, "train_batch_size_per_learner": 200},
            "sac": {"actor_lr": 1e-3, "critic_lr": 1e-3,
                    "num_steps_sampled_before_learning_starts": 200,
                    "training_intensity": 32},
        },
    },
    "thorough": {
        "model": {},  # RLlib's default, 2 x 256
        "budget": {
            "ppo": {k: 20_000 for k in range(1, 6)},
            "appo": {k: 20_000 for k in range(1, 6)},
            "sac": {k: 10_000 for k in range(1, 6)},
        },
        "training": {
            "ppo": {"lr": 3e-4, "train_batch_size_per_learner": 1000},
            "appo": {"lr": 3e-4, "train_batch_size_per_learner": 500},
            "sac": {"actor_lr": 3e-4, "critic_lr": 3e-4,
                    "num_steps_sampled_before_learning_starts": 500,
                    "training_intensity": 64},
        },
    },
}

# number: (name, environment, discount, agent names whose sorted order differs)
PROBES = {
    1: ("Constant", ConstantProbe, 0.99, False),
    2: ("Observation", ObservationProbe, 0.99, False),
    3: ("Discount", DiscountProbe, 0.5, False),
    4: ("Action", ActionProbe, 0.99, True),
    5: ("Obs -> action", ObsActionProbe, 0.99, True),
}
RUNS_DIR = ROOT / "runs" / f"probes_{PRESET}"

### The functions every probe section uses

Train the selected combinations on one probe (in parallel, reusing finished runs),
then compare each run's networks with the probe's exact answers.

In [5]:
def make_run(algorithm: str, paradigm: str, probe: int, budget_scale: float = 1.0) -> Run:
    """One training run of the selected preset."""
    name, env_class, gamma, scramble = PROBES[probe]
    preset = PRESETS[PRESET]
    suffix = "" if budget_scale == 1.0 else f"_x{budget_scale:g}"
    return Run(
        algorithm,
        paradigm,
        env_class,
        {"n_agents": 3, "scramble_ids": scramble},
        budget=int(preset["budget"][algorithm][probe] * budget_scale),
        seed=SEED,
        eval_episodes=5,
        training=preset["training"][algorithm] | {"gamma": gamma},
        model=preset["model"],
        out_dir=RUNS_DIR / f"{algorithm}_{paradigm}_p{probe}{suffix}",
    )


def status(run: Run) -> str:
    path = Path(run.out_dir) / "run.json"
    return json.loads(path.read_text())["status"] if path.exists() else "not run"


def train_all(runs: list[Run], workers: int = WORKERS) -> None:
    """Train the runs that have not finished, in parallel processes."""
    if RETRAIN:
        for run in runs:
            shutil.rmtree(run.out_dir, ignore_errors=True)
    pending = [r for r in runs if status(r) != "completed"]
    if not pending:
        return
    slowest_first = {"sac": 0, "appo": 1, "ppo": 2}
    pending.sort(key=lambda r: (slowest_first[r.algorithm], r.paradigm != "dtde"))
    start = time.time()
    with ProcessPoolExecutor(max_workers=min(workers, len(pending))) as pool:
        futures = {pool.submit(train, run): run for run in pending}
        for future in as_completed(futures):
            run = futures[future]
            try:
                note = f"trained in {future.result()['duration_s']:.0f} s"
            except Exception as error:
                note = f"FAILED: {error!r}"[:160]
            print(f"{Path(run.out_dir).name:14s} {note}", flush=True)
    print(f"{len(pending)} runs, {time.time() - start:.0f} s")


def soft_bonus(policy, run_dir: Path, module_id: str, obs: np.ndarray) -> float:
    """alpha times the policy's entropy in a state: what SAC's value adds for it."""
    metrics = pd.read_csv(Path(run_dir) / "metrics.csv")
    alpha = float(metrics[f"{module_id}/alpha_value"].dropna().iloc[-1])
    module = policy.modules[module_id]
    x = torch.as_tensor(obs, dtype=torch.float32).reshape(1, -1).repeat(4096, 1)
    with torch.no_grad():
        logits = module.forward_exploration({Columns.OBS: x})[Columns.ACTION_DIST_INPUTS]
        dist = module.get_exploration_action_dist_cls().from_logits(logits)
        entropy = float(-dist.logp(dist.sample()).mean())
    return alpha * entropy


def score(run: Run, probe: int) -> pd.DataFrame:
    """One row per checked output of a finished run: expected, got, error, ok."""
    _, env_class, gamma, _ = PROBES[probe]
    policy, _ = load_policy(run.out_dir)
    env = env_class(run.env_config)
    rows = []
    for check in env.checks(gamma):
        if check.value is not None:
            got = policy.values(check.obs)
            expected = (
                dict(check.value)
                if run.paradigm != "ctce"  # DTDE, CTDE: each agent's own value
                else {GROUP: sum(check.value.values())}
            )
            if run.algorithm == "sac" and check.name == "first step":
                after = {a: np.ones(1, np.float32) for a in env.agents_id}
                for mid in expected:
                    obs = {
                        "dtde": after,
                        "ctde": central_obs(after, env.agents_id),
                        "ctce": {GROUP: join(after, env.agents_id)},
                    }[run.paradigm][mid]
                    expected[mid] += gamma * soft_bonus(policy, run.out_dir, mid, obs)
            for mid, value in expected.items():
                rows.append((check.name, f"value {mid}", value, got[mid],
                             TOLERANCE * max(1.0, abs(value))))
        if check.action is not None:
            got = policy.act(check.obs)
            for agent, value in check.action.items():
                rows.append((check.name, f"action {agent}", value, float(got[agent][0]),
                             TOLERANCE))
    frame = pd.DataFrame(rows, columns=["state", "output", "expected", "got", "tolerance"])
    frame["error"] = (frame.got - frame.expected).abs()
    frame["ok"] = frame.error <= frame.tolerance
    return frame


def label(run: Run) -> str:
    return f"{run.algorithm.upper()} {run.paradigm.upper()}"


def check_probe(probe: int, budget_scale: float = 1.0) -> dict:
    """Train the selection on one probe and show the networks against the answers."""
    runs = [make_run(a, p, probe, budget_scale) for a in ALGORITHMS for p in PARADIGMS]
    train_all(runs)
    scores = {label(r): score(r, probe) for r in runs if status(r) == "completed"}
    failed = [label(r) for r in runs if status(r) != "completed"]
    if failed:
        print("did not finish:", ", ".join(failed))
    if not scores:
        return scores
    verdict = pd.DataFrame({
        name: {"result": "PASS" if f.ok.all() else "FAIL",
               "worst error / tolerance": (f.error / f.tolerance).max()}
        for name, f in scores.items()
    }).T
    display(verdict)
    first = next(iter(scores.values()))
    outputs = first.set_index(["state", "output"])[["expected"]]
    if all(len(f) == len(first) for f in scores.values()):
        got = pd.DataFrame({name: f.set_index(["state", "output"]).got
                            for name, f in scores.items()})
        display(outputs.join(got))
    else:  # DTDE checks each agent's value, CTCE and CTDE the group's
        for name, f in scores.items():
            print(name)
            display(f.set_index(["state", "output"])[["expected", "got", "ok"]])
    return scores


RESULTS = {}  # probe -> {combination: checks}

## Optional: train every probe at once

The probes are trained independently; only reading them is sequential. This cell
puts the selection on every probe (up to 45 runs) into one pool of workers, so a
large machine stays busy across probes instead of one probe's 9 runs at a time.
The probe sections below then find the runs finished and only score them.

- **One Ray instance for all runs.** The cell starts Ray once in this kernel and
  sets `RAY_ADDRESS`, so each run joins it instead of starting its own
  (`core/rl/train.py`). On its own, every run's Ray instance cost about 175
  threads even with its thread pools set to one, which capped the cluster at 16
  runs per node (user thread limit 4096). Training itself is unchanged: with
  `num_env_runners=0` each run samples and learns in its own process, and Ray only
  coordinates. A failure of the shared instance stops every run at once.
- The runs' processes are started with `spawn`: a forked copy of a kernel that
  has started Ray cannot use or restart it.
- On a SLURM job (the cluster) the workers are one per physical core less one
  (SLURM counts two hyper-threads per core: `-c 78` gives 38), capped by the
  thread limit at about 80 threads per run. Elsewhere it uses `WORKERS`.
- Each run uses one CPU thread for torch (the networks are small).
- Keep `RETRAIN = False` when using this cell: with `True`, every probe section
  below would delete and train its runs again.

In [ ]:
import multiprocessing

import ray

ONE_THREAD = [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "RAY_num_server_call_thread",
    "RAY_gcs_server_rpc_server_thread_num",
    "RAY_gcs_server_rpc_client_thread_num",
    "RAY_num_grpc_internal_threads",
    "RAY_worker_num_grpc_internal_threads",
    "RAY_object_manager_rpc_threads_num",
]
for name in ONE_THREAD:
    os.environ[name] = "1"
torch.set_num_threads(1)

cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", 0))
workers = WORKERS
if cpus:
    import resource

    thread_limit = resource.getrlimit(resource.RLIMIT_NPROC)[0]
    workers = min(cpus // 2 - 1, (thread_limit - 1000) // 80)

# One Ray instance for every run: train() joins it through RAY_ADDRESS. The runs'
# processes are started fresh (spawn): a forked copy of this kernel would inherit
# its Ray connection in a state that can neither be used nor replaced.
multiprocessing.set_start_method("spawn", force=True)
if not ray.is_initialized():
    ray.init(num_cpus=1, include_dashboard=False, object_store_memory=200 * 2**20)
os.environ["RAY_ADDRESS"] = ray.get_runtime_context().gcs_address

runs = [make_run(a, p, k) for k in PROBES for a in ALGORITHMS for p in PARADIGMS]
print(f"{len(runs)} runs, {workers} workers")
train_all(runs, workers=workers)

## c. Probe 1: Constant

Observation 0, reward +1, one step, the action ignored. The answer is V = 1 per
agent (3 for the group). Nothing depends on the observation or the action, so only
the value side is tested: the reward reaches the learner, the episode end is
treated as final, and the value loss and optimiser fit a constant.

In [8]:
RESULTS[1] = check_probe(1)

sac_ctce_p1    trained in 503 s
sac_ctde_p1    trained in 591 s
appo_ctce_p1   trained in 114 s
appo_dtde_p1   trained in 234 s
appo_ctde_p1   trained in 124 s
ppo_ctce_p1    trained in 225 s
sac_dtde_p1    trained in 1203 s
ppo_dtde_p1    trained in 543 s
ppo_ctde_p1    trained in 251 s
9 runs, 1367 s


,result,worst error / tolerance
PPO CTCE,PASS,0.0
PPO CTDE,PASS,0.0
PPO DTDE,PASS,0.0
APPO CTCE,PASS,0.079
APPO CTDE,PASS,0.105
APPO DTDE,PASS,0.062
SAC CTCE,PASS,0.018
SAC CTDE,PASS,0.016
SAC DTDE,PASS,0.019


PPO CTCE


,,expected,got,ok
state,output,,,
obs 0,value group,3.0,3.0,True


PPO CTDE


...

,,expected,got,ok
state,output,,,
obs 0,value group,3.0,2.992,True


SAC CTDE


,,expected,got,ok
state,output,,,
obs 0,value group,3.0,2.993,True


SAC DTDE


expected    got    ok
state output                               
obs 0 value agent_00       1.0  0.997  True
      value agent_01       1.0  0.997  True
      value agent_02       1.0  0.998  True

## d. Probe 2: Observation

Observation +1 or −1 per agent, at random; reward = the agent's own
observation. The answer is V(o) = o per agent, and the sum for the group, checked
in three states: all +1, all −1, and mixed (+1, −1, +1). A critic that ignores its
input, or reads one agent's observation for another, gets the mixed state wrong.

In [9]:
RESULTS[2] = check_probe(2)

sac_ctce_p2    trained in 398 s
sac_ctde_p2    trained in 440 s
appo_ctce_p2   trained in 97 s
appo_dtde_p2   trained in 191 s
appo_ctde_p2   trained in 132 s
ppo_ctce_p2    trained in 283 s
sac_dtde_p2    trained in 1071 s
ppo_dtde_p2    trained in 609 s
ppo_ctde_p2    trained in 262 s
9 runs, 1270 s


,result,worst error / tolerance
PPO CTCE,PASS,0.0
PPO CTDE,PASS,0.0
PPO DTDE,PASS,0.0
APPO CTCE,PASS,0.083
APPO CTDE,PASS,0.277
APPO DTDE,PASS,0.076
SAC CTCE,PASS,0.024
SAC CTDE,PASS,0.032
SAC DTDE,PASS,0.032


PPO CTCE


,,expected,got,ok
state,output,,,
all +1,value group,3.0,3.0,True
all -1,value group,-3.0,-3.0,True
mixed,value group,1.0,1.0,True


PPO CTDE


...

,,expected,got,ok
state,output,,,
all +1,value group,3.0,3.001,True
all -1,value group,-3.0,-3.011,True
mixed,value group,1.0,0.998,True


SAC CTDE


,,expected,got,ok
state,output,,,
all +1,value group,3.0,3.002,True
all -1,value group,-3.0,-3.012,True
mixed,value group,1.0,0.995,True


SAC DTDE


expected    got    ok
state  output                               
all +1 value agent_00       1.0  0.995  True
       value agent_01       1.0  1.003  True
       value agent_02       1.0  0.996  True
all -1 value agent_00      -1.0 -1.002  True
       value agent_01      -1.0 -0.997  True
       value agent_02      -1.0 -1.000  True
mixed  value agent_00       1.0  0.995  True
       value agent_01      -1.0 -0.997  True
       value agent_02       1.0  0.996  True

## e. Probe 3: Discount

Two steps: observation 0 then 1, reward 0 then +1. With γ = 0.5, V(1) = 1 and
V(0) = γ · 1 = 0.5 per agent (1.5 and 3 for the group). Tests bootstrapping from
the next state, the discount, and where the episode ends. SAC's first-step answer
adds its entropy bonus (see section a).

In [10]:
RESULTS[3] = check_probe(3)

sac_ctce_p3    trained in 424 s
sac_ctde_p3    trained in 472 s
appo_ctce_p3   trained in 78 s
appo_dtde_p3   trained in 146 s
appo_ctde_p3   trained in 94 s
ppo_ctce_p3    trained in 154 s
ppo_dtde_p3    trained in 381 s
sac_dtde_p3    trained in 1011 s
ppo_ctde_p3    trained in 200 s
9 runs, 1052 s


,result,worst error / tolerance
PPO CTCE,PASS,0.333
PPO CTDE,PASS,0.333
PPO DTDE,PASS,0.167
APPO CTCE,PASS,0.049
APPO CTDE,PASS,0.145
APPO DTDE,PASS,0.055
SAC CTCE,PASS,0.026
SAC CTDE,PASS,0.038
SAC DTDE,PASS,0.044


PPO CTCE


,,expected,got,ok
state,output,,,
first step,value group,1.5,1.425,True
last step,value group,3.0,3.000,True


PPO CTDE


...

,,expected,got,ok
state,output,,,
first step,value group,2.002,1.994,True
last step,value group,3.000,2.992,True


SAC CTDE


,,expected,got,ok
state,output,,,
first step,value group,2.002,2.000,True
last step,value group,3.000,2.983,True


SAC DTDE


expected    got    ok
state      output                               
first step value agent_00     0.667  0.666  True
           value agent_01     0.668  0.672  True
           value agent_02     0.667  0.673  True
last step  value agent_00     1.000  1.003  True
           value agent_01     1.000  1.000  True
           value agent_02     1.000  0.999  True

## f. Probe 4: Action

Observation 0; reward 1 − (a − t)² with a different target per agent (−0.6, 0,
0.6) and agent names that sort differently. The answer is each agent's action =
its own target. Tests the policy update and that each agent's action reaches its
own reward. SAC's means sit a little inside the targets (entropy bonus, tanh
squashing).

In [ ]:
# FAILED: Exception('The current node timed out during startup. This could happen because some of the raylet failed to startup or the GCS has become overloaded.')
RESULTS[4] = check_probe(4)

In [11]:
RESULTS[4] = check_probe(4)

sac_ctce_p4    trained in 413 s
sac_ctde_p4    trained in 462 s
appo_ctce_p4   trained in 94 s
appo_dtde_p4   trained in 195 s
appo_ctde_p4   trained in 120 s
ppo_ctce_p4    trained in 225 s
ppo_ctde_p4    FAILED: Exception('The current node timed out during startup. This could happen because some of the raylet failed to startup or the GCS has become overloaded.')
sac_dtde_p4    trained in 1024 s
ppo_dtde_p4    trained in 496 s
9 runs, 1141 s
did not finish: PPO CTDE


,result,worst error / tolerance
PPO CTCE,PASS,0.773
PPO DTDE,PASS,0.555
APPO CTCE,PASS,0.178
APPO CTDE,PASS,0.791
APPO DTDE,FAIL,1.022
SAC CTCE,PASS,0.572
SAC CTDE,PASS,0.828
SAC DTDE,PASS,0.72


expected  PPO CTCE  PPO DTDE  APPO CTCE  APPO CTDE  APPO DTDE  SAC CTCE  SAC CTDE  SAC DTDE
state output                                                                                                      
obs 0 action agent_2       -0.6    -0.716    -0.683     -0.609     -0.650     -0.638    -0.514    -0.476    -0.492
      action agent_10       0.0     0.065     0.048     -0.018     -0.027      0.016     0.010     0.020     0.012
      action agent_1        0.6     0.688     0.595      0.627      0.719      0.753     0.516     0.500     0.531

## g. Probe 5: Obs -> action

Observation +1 or −1 per agent; reward 1 − (a − o/2)². The answer is action =
o/2 (±0.5), checked in the same three states as probe 2. Tests that each agent's
policy uses its own observation, and that observation, action and reward stay
aligned through batching. CTCE's single network must learn which observation
matters for whom from the summed reward, so it is the slowest here.

In [12]:
RESULTS[5] = check_probe(5)

sac_dtde_p5    FAILED: ValueError('After taking into account object store and redis memory usage, the amount of memory on this node available for tasks and actors (0.09 GB) is
sac_ctce_p5    FAILED: ValueError('After taking into account object store and redis memory usage, the amount of memory on this node available for tasks and actors (0.09 GB) is
sac_ctde_p5    FAILED: ValueError('Attempting to cap object store memory usage at 44827852 bytes, but the minimum allowed is 78643200 bytes.')
appo_ctce_p5   trained in 138 s
appo_ctde_p5   trained in 162 s
appo_dtde_p5   trained in 238 s
ppo_ctce_p5    trained in 232 s
ppo_ctde_p5    trained in 283 s
ppo_dtde_p5    trained in 474 s
9 runs, 651 s
did not finish: SAC CTCE, SAC CTDE, SAC DTDE


,result,worst error / tolerance
PPO CTCE,FAIL,2.856
PPO CTDE,PASS,0.956
PPO DTDE,PASS,0.873
APPO CTCE,PASS,0.971
APPO CTDE,PASS,0.477
APPO DTDE,PASS,0.444


expected  PPO CTCE  PPO CTDE  PPO DTDE  APPO CTCE  APPO CTDE  APPO DTDE
state  output                                                                                  
all +1 action agent_2        0.5     0.692     0.584     0.577      0.560      0.571      0.510
       action agent_10       0.5     0.552     0.550     0.490      0.502      0.559      0.454
       action agent_1        0.5     0.816     0.572     0.480      0.478      0.536      0.567
all -1 action agent_2       -0.5    -0.715    -0.623    -0.631     -0.646     -0.494     -0.477
       action agent_10      -0.5    -0.497    -0.643    -0.466     -0.515     -0.483     -0.512
       action agent_1       -0.5    -0.512    -0.445    -0.608     -0.500     -0.474     -0.455
mixed  action agent_2        0.5     0.928     0.584     0.577      0.565      0.571      0.510
       action agent_10      -0.5    -0.686    -0.643    -0.466     -0.546     -0.483     -0.512
       action agent_1        0.5     0.626     0.572     0.480      0.471      0.536      0.567

## h. Summary of the selection

`PASS` or `FAIL` per probe and combination, with the largest error relative to its
tolerance (1.0 is the limit). Read each column top to bottom: the first failure is
the piece to look at. Probes not run yet show as empty.

In [13]:
summary = pd.DataFrame({
    combination: {
        f"{k} {PROBES[k][0]}": (f"{'PASS' if f.ok.all() else 'FAIL'} "
                                f"{(f.error / f.tolerance).max():.2f}")
        for k, scores in sorted(RESULTS.items()) for c, f in scores.items() if c == combination
    }
    for combination in [f"{a.upper()} {p.upper()}" for a in ALGORITHMS for p in PARADIGMS]
})
summary

,PPO CTCE,PPO CTDE,PPO DTDE,APPO CTCE,APPO CTDE,APPO DTDE,SAC CTCE,SAC CTDE,SAC DTDE
1 Constant,PASS 0.00,PASS 0.00,PASS 0.00,PASS 0.08,PASS 0.11,PASS 0.06,PASS 0.02,PASS 0.02,PASS 0.02
2 Observation,PASS 0.00,PASS 0.00,PASS 0.00,PASS 0.08,PASS 0.28,PASS 0.08,PASS 0.02,PASS 0.03,PASS 0.03
3 Discount,PASS 0.33,PASS 0.33,PASS 0.17,PASS 0.05,PASS 0.15,PASS 0.06,PASS 0.03,PASS 0.04,PASS 0.04
4 Action,PASS 0.77,NaN,PASS 0.55,PASS 0.18,PASS 0.79,FAIL 1.02,PASS 0.57,PASS 0.83,PASS 0.72
5 Obs -> action,FAIL 2.86,PASS 0.96,PASS 0.87,PASS 0.97,PASS 0.48,PASS 0.44,NaN,NaN,NaN


## i. Learner health

Per run: entropy at the start and end (PPO, APPO; it should fall as the policy
sharpens, never grow without limit), SAC's temperature at the end, and the run's
status (a NaN loss stops a run, so it shows as failed).

In [14]:
def health(run: Run) -> dict:
    row = {"status": status(run)}
    path = Path(run.out_dir) / "metrics.csv"
    if path.exists():
        metrics = pd.read_csv(path)
        entropy = metrics.filter(like="/entropy").mean(axis=1).dropna()
        alpha = metrics.filter(like="/alpha_value").mean(axis=1).dropna()
        if len(entropy):
            row |= {"entropy start": entropy.iloc[0], "entropy end": entropy.iloc[-1]}
        if len(alpha):
            row |= {"alpha end": alpha.iloc[-1]}
        row["env steps"] = int(metrics.env_steps.iloc[-1])
    return row


pd.DataFrame({
    (a.upper(), p.upper(), k): health(make_run(a, p, k))
    for a in ALGORITHMS for p in PARADIGMS for k in PROBES
}).T

status entropy start entropy end env steps alpha end
PPO  CTCE 1  completed         4.157       2.006     20000       NaN
          2  completed         4.239      64.257     20000       NaN
          3  completed         4.273       2.164     20000       NaN
          4  completed         3.852      -0.038     20000       NaN
          5  completed         3.927       0.103     20000       NaN
     CTDE 1  completed         3.805       1.182     20000       NaN
          2  completed         6.747      64.257     20000       NaN
          3  completed        10.009       8.573     20000       NaN
          4    running           NaN         NaN       NaN       NaN
          5  completed         3.692      -0.475     20000       NaN
     DTDE 1  completed         1.422      -0.346     20000       NaN
          2  completed         4.084      21.419     20000       NaN
          3  completed         2.504       0.977     20000       NaN
          4  completed         1.192      -1.536     20000       NaN
          5  completed         1.221      -1.566     20000       NaN
APPO CTCE 1  completed         3.984       8.345     20000       NaN
          2  completed         5.076       7.559     20000       NaN
          3  completed         4.651       7.158     20000       NaN
          4  completed         2.718      -0.575     20000       NaN
          5  completed         3.066      -0.255     20000       NaN
     CTDE 1  completed         4.067      10.194     20000       NaN
          2  completed         6.718      12.379     20000       NaN
          3  completed         5.927      15.656     20000       NaN
          4  completed         2.395      -0.046     20000       NaN
          5  completed         2.838        0.14     20000       NaN
     DTDE 1  completed         2.632       3.928     20000       NaN
          2  completed         1.738       3.855     20000       NaN
          3  completed          1.98       4.453     20000       NaN
          4  completed         0.573      -0.542     20000       NaN
          5  completed         0.851      -0.684     20000       NaN
SAC  CTCE 1  completed           NaN         NaN     10000      0.49
          2  completed           NaN         NaN     10000      0.49
          3  completed           NaN         NaN     10000      0.49
          4  completed           NaN         NaN     10000      0.49
          5    running           NaN         NaN       NaN       NaN
     CTDE 1  completed           NaN         NaN     10000      0.49
          2  completed           NaN         NaN     10000      0.49
          3  completed           NaN         NaN     10000      0.49
          4  completed           NaN         NaN     10000      0.49
          5    running           NaN         NaN       NaN       NaN
     DTDE 1  completed           NaN         NaN     10000      0.49
          2  completed           NaN         NaN     10000      0.49
          3  completed           NaN         NaN     10000      0.49
          4  completed           NaN         NaN     10000      0.49
          5    running           NaN         NaN       NaN       NaN

## j. Follow-up: a longer budget

When a combination fails a probe, train it again with a multiple of the budget to
separate "slow" from "broken": a slow learner keeps closing the gap with more steps,
a broken one does not. For example CTCE on probe 5: one network learns every
agent's action from the summed reward, so the signal for one agent carries the
other agents' reward noise (L-02 in the validation document).

In [15]:
# e.g. check_probe(5, budget_scale=3) for the selection, or narrow it first:
# ALGORITHMS, PARADIGMS = ["ppo"], ["ctce"]; check_probe(5, budget_scale=3)